# Day 2 · Module 2 — Variant Effect Prediction (Zero-Shot VEP Scorer)

**Driving question:** *Can a model that has never seen a single clinical diagnosis still tell a harmless DNA change from a disease-causing one — just by being surprised?*

**Objective:** turn a DNA language model's likelihoods into a variant-effect score (`delta = LL(alt) − LL(ref)`), validate it honestly against real labels with AUROC, and see for yourself why **scale** decides whether the method works.

> **Instructor solution** · kernel **AImed**. No pass/fail gate — if you get stuck, the CHECKPOINT cell near the end plots the answer and the caption tells you how to read it. Everything here runs offline on CPU.

### Pre-work recap
- A DNA language model assigns a **log-likelihood** to any A/C/G/T string — higher means "this looks like real, natural genome." (You built this `sequence_logprob` helper in Module 1.)
- **Zero-shot** = predicting *without ever training on a clinical label*. The only "supervision" is the genomes the model was pretrained on.
- The whole method is one subtraction: `delta = LL(alt_window) − LL(ref_window)`. More negative → the mutation surprised the model more → more likely disruptive.
- *Arrive-with:* **why** can a model that saw zero clinical labels predict pathogenicity? (Hold that thought for the Background cell.)
- Data: bundled **BRCA1** set (`brca1_variants.csv`, 3,893 single-letter variants; binary `is_lof` label = does this break the gene?) and the bundled **chr17** reference (hg19/GRCh37 build).

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import clinical_risk_console` and `from common import metrics`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Guided hands-on

### Background — what a "zero-shot variant-effect scorer" actually is

New to this? Read first — the whole module is **one subtraction**, but it helps to know what is being subtracted.

**The model.** A *DNA language model* (we use **HyenaDNA-small**, the small one you ran in Module 1; the frontier one is **Evo 2**) was pretrained on huge piles of real genome with one job: predict the next DNA letter. As a side effect it learns what "normal" genome looks like. Given any string of A/C/G/T it returns a **log-likelihood** — a single number for *how natural this sequence looks to me*. Higher = looks like real genome; lower = looks weird/surprising.

**The trick (this is the entire method).** Take a window of reference DNA around a variant's position. Make a copy with the one letter swapped to the patient's `alt` letter. Score **both** with the model and subtract:

```
delta = LL(alt_window) − LL(ref_window)
```

- `delta ≈ 0` → the model is just as comfortable with the mutated sequence → probably a harmless change.
- `delta` strongly **negative** → the mutation made the sequence look *unnatural* to the model → the kind of change evolution rarely tolerates → more likely **pathogenic**.

**Why does this work with zero clinical labels?** This is the key idea of the whole day. The model never saw the word "pathogenic." Its only teacher was **evolution**: over millions of years, mutations that break important genes got selected *out* of the genomes the model trained on, so the model rarely saw them and finds them surprising. The clinical signal leaks in through the **conservation** baked into the training genomes. That is the "implicit supervision."

**Analogy — a copy-editor who never read the rulebook.** Imagine an editor who has read millions of well-written English sentences but was *never taught grammar rules*. Hand them "The cat sat on the mat" → they shrug, looks fine. Change it to "The cat sat on the *zat*" → they flinch: *that's not a word I've ever seen here*. They can't quote a rule, but their flinch reliably flags the broken sentence. `delta` is the size of the model's flinch. A big flinch on a DNA edit is our zero-shot guess that the edit is damaging — no medical training required.

**Validating it honestly (the Day-1 move, on genomes).** A `delta` per variant is just a *ranking score*. To check it, line all 3,893 variants up by risk and ask: do the truly broken ones (`is_lof == 1`) tend to land near the top? That is exactly **AUROC** — the chance a random truly-broken variant outranks a random harmless one (0.5 = coin flip, 1.0 = perfect). We compute `auroc(is_lof, -delta)`. The minus sign matters and you'll discover why in YOUR TURN.

**Key terms.** *log-likelihood (LL)* = the model's "how natural" score · *zero-shot* = no clinical labels used · *delta* = LL(alt) − LL(ref) · *is_lof* = loss-of-function, the 0/1 "does it break the gene" label · *AUROC* = ranking quality · *hg19* = the genome build the BRCA1 coordinates use (must match the reference we slice from).

### Build the scorer, then validate it — worked example

The backing script `vep_scorer.py` already wires the pieces together: `score_variant(ref, alt, scorer)` is *literally* `scorer.sequence_logprob(alt) - scorer.sequence_logprob(ref)`, and `run_vep(...)` builds every BRCA1 window off chr17, scores them, and reports AUROC. We let it do the heavy lifting and **print the key numbers**. On CPU with no GPU it transparently falls back to the precomputed score table — same numbers, no waiting.

**Want to run it live yourself?** Go for it — `run_vep(use_precomputed=False)` (CLI: `python scripts/vep_scorer.py --device cpu`) runs the model on every ref/alt window and reproduces the deltas, so you can check the numbers for yourself. It needs the **full chr17 reference** (~25 MB), which is too big to ship in a `git clone`: download hg19 chr17 from [UCSC goldenPath](https://hgdownload.soe.ucsc.edu/goldenPath/hg19/chromosomes/chr17.fa.gz) and save it as `data/day2_genomics/GRCh37.p13_chr17.fna.gz` (on Midway3 it's already staged there). **For the sake of time, everything below uses the precomputed score table** (`run_vep(use_precomputed=True)`) — identical numbers, no model, no GPU, no download.

In [ ]:
# Guided compute — score BRCA1 with the SMALL model and validate it honestly.
import inspect
from vep_scorer import run_vep, score_variant, evo2_auroc

# `score_variant` IS the whole method -- two likelihood calls and a subtraction:
print('score_variant source:')
print(inspect.getsource(score_variant))

# Run the small model over a BRCA1 subset and validate against the is_lof labels.
# (Module 1 showed the model scoring DNA live; scoring all 400 BRCA1 windows on CPU is slow, so we
# read the small model's precomputed BRCA1 deltas here -- identical numbers, instant.)
df, hyena_auroc, info = run_vep(use_precomputed=True, n=400, log=lambda *a: None)
print(f"\nsource used   : {info['source']}")
print(f"HyenaDNA-small: n={len(df)}  AUROC = {hyena_auroc:.3f}")
print(f"  mean delta   benign={df.loc[df.is_lof==0,'delta'].mean():+.3f}   "
      f"LoF={df.loc[df.is_lof==1,'delta'].mean():+.3f}   (LoF should be lower)")

evo2 = evo2_auroc()   # AUROC straight from the REAL Evo 2 7B precomputed table (or None)
print(f"\nEvo 2 7B (frontier, precomputed table): "
      + (f'AUROC = {evo2:.3f}' if evo2 is not None else 'table not generated here'))
print('Read it: the METHOD is identical for both models. Only the model changed.')

### The vivid demo — same method, two models, on your own BRCA1 data
The method is one subtraction. So if HyenaDNA-small lands near 0.5 (coin flip) and Evo 2 lands near 0.88 on the **exact same variants with the exact same code**, the difference cannot be the method — it must be the model. The cell below puts both score columns side by side and ranks the variants by each. Watch where the truly-broken (`is_lof==1`) variants pile up.

In [ ]:
# VIVID DEMO -- identical pipeline, swap only the model. Uses the precomputed tables
# (both carry a delta column) so it runs offline on CPU in a blink.
import pandas as pd, numpy as np
from common import metrics as M

pre = pd.read_csv(cfg.PRECOMPUTED_VEP_EVO2)   # has BOTH 'hyena_delta' and 'evo2_delta_score'
y = pre['is_lof'].to_numpy()

for label, col in [('HyenaDNA-small', 'hyena_delta'), ('Evo 2 7B (frontier)', 'evo2_delta_score')]:
    risk = -pre[col].to_numpy()                      # negate: more-negative delta -> higher risk
    auroc = M.auroc(y, risk)
    # of the 25 variants this model calls RISKIEST, how many are truly loss-of-function?
    top25 = pre.iloc[np.argsort(-risk)[:25]]
    hit = int(top25['is_lof'].sum())
    print(f'{label:22s}  AUROC={auroc:.3f}   |   top-25 riskiest: {hit}/25 are truly LoF')

base_rate = y.mean()
print(f'\n(baseline: only {base_rate:.0%} of ALL variants are LoF, so 25 random picks would catch ~{25*base_rate:.0f})')
print('Undeniable: same code, same variants. HyenaDNA ~ coin flip; Evo 2 sorts the broken ones to the top.')

### Why the minus sign? (read before YOUR TURN)
AUROC needs scores oriented so that **higher = more likely positive** (here, positive = `is_lof == 1`, broken). But our `delta` is the opposite: a *more negative* delta means *more* disruptive. So we feed AUROC the **negated** delta, `-delta`, to flip it into a risk score. Get the sign backwards and AUROC doesn't go to chance — it goes to `1 - AUROC`, the mirror image. You'll prove that next.

In [ ]:
# YOUR TURN -- compute the AUROC yourself, both orientations, and explain the mirror.
# Fill each  = ...  (the solution notebook has the answers).
import numpy as np
import pandas as pd
from common import metrics as M

pre = pd.read_csv(cfg.PRECOMPUTED_VEP_EVO2)
y      = pre['is_lof'].to_numpy()
delta  = pre['evo2_delta_score'].to_numpy()    # Evo 2's raw delta = LL(alt) - LL(ref)

# (a) The CORRECT orientation: risk score = the NEGATED delta (more-negative delta -> higher risk).
risk_correct = -delta                              # <-- fill in (hint: a minus sign)
auroc_correct = M.auroc(y, risk_correct)

# (b) The WRONG orientation: feed the raw delta straight in, no negation.
auroc_wrong = M.auroc(y, delta)                   # <-- fill in (the un-negated delta)

print(f'(a) correct orientation  AUROC = {auroc_correct:.3f}')
print(f'(b) wrong   orientation  AUROC = {auroc_wrong:.3f}')
print(f'    check: wrong == 1 - correct ?  {abs(auroc_wrong - (1 - auroc_correct)) < 1e-6}')
print('Lesson: orientation never destroys the information -- it mirrors AUROC about 0.5.')
print('A model at 0.12 is NOT useless; it is a great predictor read upside-down.')

In [ ]:
# VISUAL -- the two delta distributions, the original "aha" of zero-shot VEP.
# If the broken (LoF) variants sit at MORE NEGATIVE delta than the harmless ones,
# the scorer separates them with zero labels used in scoring.
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd, numpy as np
from common import metrics as M

pre = pd.read_csv(cfg.PRECOMPUTED_VEP_EVO2)
fig, (axS, axB) = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, col, title in [(axS, 'hyena_delta', 'HyenaDNA-small (the model you ran)'),
                       (axB, 'evo2_delta_score', 'Evo 2 7B (the frontier model)')]:
    ben = pre.loc[pre.is_lof == 0, col]
    lof = pre.loc[pre.is_lof == 1, col]
    lo, hi = np.percentile(pre[col], [1, 99])
    bins = np.linspace(lo, hi, 40)
    ax.hist(ben, bins=bins, alpha=0.6, color='C0', density=True, label='benign (is_lof=0)')
    ax.hist(lof, bins=bins, alpha=0.6, color='C3', density=True, label='loss-of-function (is_lof=1)')
    ax.set(xlabel='delta = LL(alt) - LL(ref)   (more negative ->)',
           ylabel='density', title=f'{title}\nAUROC={M.auroc(pre.is_lof, -pre[col]):.3f}')
    ax.legend()
plt.tight_layout(); plt.show()
print('Read it: when the red (LoF) hump sits LEFT of the blue (benign) hump, the model is')
print('flinching at the damaging variants. HyenaDNA: humps overlap. Evo 2: red shifts left.')

### Your task — decide what an Evo 2 score is allowed to do for a patient
Evo 2 reaches **AUROC ≈ 0.88** on this *population* of BRCA1 variants — genuinely useful for ranking. But a clinician does not have a population; they have **one patient** holding one variant. Using the numbers above (and the CADD baseline you'll see in the checkpoint), fill in the **`DECISION`** dict: would you let this score *rule a variant pathogenic on its own*, *rule it benign*, or only *prioritize* which variants a human expert reviews first? Defend each choice with a specific number. There is no single right answer — what matters is the reasoning, argued in the discussion.

In [ ]:
# A defended deployment call (yours may differ -- what matters is the reasoning, cite numbers).
DECISION = {
    'use_alone_to_call_pathogenic': False,
    'best_clinical_role': 'triage / prioritize variants for expert review (a prior, not a verdict)',
    'population_vs_patient': ('AUROC ~0.88 is a POPULATION ranking statistic -- it promises good '
                             'average separation, not certainty about any ONE patient variant'),
    'small_model_verdict': ('no -- HyenaDNA-small is ~chance on BRCA1 (AUROC ~0.46); it is the honest '
                            'small-model baseline, not deployable. The frontier model (Evo 2, ~0.88) is'),
}
print(DECISION)

### Checkpoint — stuck? Visualize the decision
Couldn't commit to a role for the score? Run the next cell. It plots **how AUROC depends on the variant type** and compares Evo 2 against a classic non-deep-learning baseline (CADD), so you can read your `best_clinical_role` straight off the figure. Non-blocking; uses only the precomputed table.

In [ ]:
# CHECKPOINT -- stuck on the decision? This VISUALIZES it. Offline, CPU, precomputed table.
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd, numpy as np
from common import metrics as M

pre = pd.read_csv(cfg.PRECOMPUTED_VEP_EVO2)
y = pre['is_lof'].to_numpy()
evo2_all = M.auroc(y, -pre['evo2_delta_score'].to_numpy())
cadd_all = M.auroc(y,  pre['CADD.score'].to_numpy())   # CADD: higher already = more damaging

# AUROC within each consequence type that has both classes (where the easy/hard cases live)
rows = []
for c in pre['consequence'].value_counts().index:
    sub = pre[pre.consequence == c]
    if sub.is_lof.nunique() == 2 and len(sub) >= 30:
        rows.append((c, len(sub), M.auroc(sub.is_lof, -sub['evo2_delta_score'])))
rows.sort(key=lambda r: r[2])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.2))
axL.bar(['Evo 2\n(all)', 'CADD\nbaseline'], [evo2_all, cadd_all], color=['C3', '0.6'])
axL.axhline(0.5, color='k', ls='--', lw=1, label='coin flip')
axL.set(ylim=(0, 1), ylabel='AUROC', title='Evo 2 vs a classic baseline (whole population)')
axL.legend()
axR.barh([r[0] for r in rows], [r[2] for r in rows], color='C3')
axR.axvline(0.5, color='k', ls='--', lw=1)
axR.set(xlim=(0, 1), xlabel='AUROC within this variant type', title='Where the score is strong vs shaky')
plt.tight_layout(); plt.show()
print(f'Evo 2 overall AUROC = {evo2_all:.3f}  |  CADD baseline = {cadd_all:.3f}')
print('Read it: Evo 2 beats the classic baseline overall, but per-type bars near 0.5 are where')
print('the score is shaky -- exactly the variants you would NOT call on the score alone.')

**How to read the checkpoint plot.**

*Left — Evo 2 vs CADD (a sanity check the score is real).* CADD is a well-established, non-deep-learning variant scorer. Evo 2's bar clearing CADD's bar — and both clearing the dashed coin-flip line — says the zero-shot delta is carrying *genuine* signal, not an artifact. **Analogy — a new blood test must beat the existing one to earn a place.** If Evo 2 only matched a coin flip, no role would be defensible; that it beats an established tool is what *earns it a seat at the table*.

*Right — AUROC broken down by variant type (where to trust it, where not to).* Each bar is the score's ranking quality *within one kind of variant*. **Analogy — a metal detector that beeps reliably over coins but barely over foil.** Bars near 1.0 are variant types where the score is trustworthy; bars hovering near the 0.5 dashed line are types where the score is **near a coin flip** — the model is essentially guessing there. That split is the heart of your `DECISION`: a score that is strong on the whole population but coin-flippy on *your patient's particular kind of variant* is exactly why you let it **prioritize expert review**, not **make the call alone**. Read your `best_clinical_role` off this: a tool that is strong overall but unreliable on some slices is a *triage* tool, not an *autonomous diagnostician*.

### Discussion (peer instruction — vote, argue with a neighbor who disagreed, re-vote)
- The scorer used **zero clinical labels** yet ranks pathogenicity at AUROC 0.88. What is the implicit "supervision," and where might it **leak or fail**? (Hint: what kinds of damaging variants would evolution *not* have selected against — and so the model would *not* find surprising?)
- HyenaDNA-small is ~coin-flip on BRCA1, yet you proved the *method* is correct (the orientation mirror, the identical pipeline). What does that teach about blaming your **code** vs blaming the **model** when a result looks bad?
- A clinician asks, *"Evo 2 scores my patient's BRCA1 variant as high-risk — so is it pathogenic?"* Given AUROC 0.88 is a **population** statistic and some variant types sit near coin-flip, what is your honest one-sentence answer?

### Stretch (optional)
1. **Beat the baseline where it lives.** Filter to coding **Missense** variants only (the hard "variant of uncertain significance" class) and recompute Evo 2's AUROC vs CADD's. Does the deep model's edge grow or shrink on the clinically hardest slice?
2. **Score something CADD can't.** Pull a **non-coding / intronic** variant from the table and compute its delta by hand with `score_variant` and a window from chr17 (`from common import genomics`; `genomics.build_window(pos, ref, alt, full_seq)`). Classic protein-based scorers can't touch these — that is exactly the frontier a sequence model extends.
3. **Find the model's worst miss.** Sort by `risk_correct` and find a truly-benign variant the model ranked near the top (a confident false alarm). Open its `clinvar` and `consequence` columns: what would it have cost to hand that single call to a patient?